In [20]:
import cv2
import time

IDX = 1  # 예전에 잡힌 인덱스로 먼저 테스트 추천!

for backend in [cv2.CAP_DSHOW, cv2.CAP_MSMF]:
    print("backend:", backend)

    cap = cv2.VideoCapture(IDX, backend)
    print("isOpened:", cap.isOpened())

    # 워밍업(최대 2초)
    t0 = time.time()
    ok = False
    while time.time() - t0 < 2.0:
        ret, frame = cap.read()
        if ret and frame is not None:
            ok = True
            break

    if not ok:
        print("FAIL: no frame")
        cap.release()
        continue

    print("SUCCESS: streaming")
    while True:
        ret, frame = cap.read()
        if not ret or frame is None:
            print("stream lost")
            break
        cv2.imshow(f"test backend {backend}", frame)
        if cv2.waitKey(1) == 27:  # ESC
            break

    cap.release()
    cv2.destroyAllWindows()

backend: 700
isOpened: True
SUCCESS: streaming
backend: 1400
isOpened: True
SUCCESS: streaming


In [18]:
import asyncio
import cv2
import av
import numpy as np
import nest_asyncio
from aiortc import RTCPeerConnection, VideoStreamTrack, RTCSessionDescription

# 1. 주피터 중첩 루프 허용
nest_asyncio.apply()

class DjiWebRTCTestTrack(VideoStreamTrack):
    """DJI Action 4 영상을 WebRTC 트랙으로 변환"""
    def __init__(self, cap_index=1):
        super().__init__()
        self.cap = cv2.VideoCapture(cap_index, cv2.CAP_DSHOW)
        # DJI 최적화 해상도 (720p)
        self.cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
        self.cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
        
        if not self.cap.isOpened():
            print(f"❌ [카메라 에러] Index {cap_index}를 열 수 없습니다. 인덱스를 확인하세요.")
        else:
            print(f"✅ [카메라 연결] DJI Action 4 준비 완료 (Index {cap_index})")

    async def recv(self):
        pts, time_base = await self.next_timestamp()
        
        # 비차단 방식으로 프레임 읽기
        ret, frame = await asyncio.to_thread(self.cap.read)
        
        if not ret or frame is None:
            # 프레임 누락 시 검은 화면 송출로 스트림 유지
            frame = np.zeros((720, 1280, 3), dtype=np.uint8)
            cv2.putText(frame, "WAITING FOR DJI...", (450, 360), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 2)
        else:
            # 화면에 테스트 텍스트 오버레이
            cv2.putText(frame, "JOY WebRTC: STREAMING", (50, 50), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        # WebRTC 표준 RGB 변환
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        new_frame = av.VideoFrame.from_ndarray(frame_rgb, format="rgb24")
        new_frame.pts = pts
        new_frame.time_base = time_base
        return new_frame

    def stop(self):
        super().stop()
        if self.cap:
            self.cap.release()
            print("🛑 DJI 카메라 자원 해제")

async def run_local_webrtc_test():
    """송신부와 수신부를 동시에 시뮬레이션"""
    pc_send = RTCPeerConnection()
    pc_recv = RTCPeerConnection()

    # 송신 측: DJI 트랙 추가
    track = DjiWebRTCTestTrack(cap_index=1)
    pc_send.addTrack(track)

    @pc_recv.on("track")
    def on_track(remote_track):
        print("🎬 [WebRTC] 트랙 수신 성공! 팝업 창을 확인하세요.")
        async def display_loop():
            try:
                while True:
                    frame = await remote_track.recv()
                    # 다시 BGR로 바꿔서 OpenCV로 출력
                    img = frame.to_ndarray(format="bgr24")
                    cv2.imshow("DJI Action 4 WebRTC Test (Press ESC to Close)", img)
                    
                    if cv2.waitKey(1) == 27: # ESC 키
                        break
            except Exception as e:
                print(f"💡 재생 루프 종료: {e}")
            finally:
                cv2.destroyAllWindows()
        
        asyncio.create_task(display_loop())

    # --- WebRTC Handshake (SDP 교환) ---
    offer = await pc_send.createOffer()
    await pc_send.setLocalDescription(offer)
    await pc_recv.setRemoteDescription(pc_send.localDescription)

    answer = await pc_recv.createAnswer()
    await pc_recv.setLocalDescription(answer)
    await pc_send.setRemoteDescription(pc_recv.localDescription)

    print("🚀 WebRTC 핸드쉐이크 완료! 스트리밍 중...")
    
    try:
        # 30초 동안 테스트 진행
        await asyncio.sleep(30)
    except asyncio.CancelledError:
        pass
    finally:
        track.stop()
        await pc_send.close()
        await pc_recv.close()
        print("✅ 테스트 프로세스 정상 종료")

# 실행
await run_local_webrtc_test()

✅ [카메라 연결] DJI Action 4 준비 완료 (Index 1)
🎬 [WebRTC] 트랙 수신 성공! 팝업 창을 확인하세요.
🚀 WebRTC 핸드쉐이크 완료! 스트리밍 중...
🛑 DJI 카메라 자원 해제
🛑 DJI 카메라 자원 해제
💡 재생 루프 종료: 
✅ 테스트 프로세스 정상 종료
